In [0]:
%run ./config

In [0]:
agg_df=spark.read.format('delta').load(gold+'/fact_table')
repayments=spark.read.format('delta').load(gold+'/repayment')

In [0]:
agg_df.createOrReplaceTempView("agg_df")
repayments.createOrReplaceTempView('repayments')

In [0]:
risk_segmentation_df=spark.sql("""
                     with cte1 as (
                         select badge, 
                                count(loan_id) no_of_loans , 
                                round(sum(outstanding_amount),2) outstanding_amount
                         from agg_df
                         group by badge
                     ),
                     cte2 as (
                         select badge, round(avg(days_past_due),2) avg_dpd, 
                                       round(sum(case when days_past_due>60 then 1 else 0 end)*100/
                                       count(*),2) as default_rate_proxy
                         from repayments r join agg_df a on r.loan_id=a.loan_id
                         group by badge
                     )
                     select c1.badge, 
                            c1.no_of_loans, 
                            c1.outstanding_amount, 
                            c2.avg_dpd, 
                            c2.default_rate_proxy
                        from cte1 c1 join cte2 c2 on c1.badge=c2.badge
                     """)


In [0]:
risk_segmentation_df.write.mode('overwrite').option('overwriteSchema', True).format('delta').save(gold+'/risk_segmentation')

In [0]:
df=spark.read.format('delta').load(gold+'/risk_segmentation')
display(df)